GOLD

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df_clientes = spark.read.table("silver.default.clientes_silver")
df_estoque = spark.read.table("silver.default.estoque_silver")
df_vendas = spark.read.table("silver.default.vendas_silver")


df_base = (
    df_vendas
    .filter(col("registro_valido") == True)
    .join(df_clientes, df_vendas.cliente_id == df_clientes.cliente_id, "inner")
    .join(df_estoque,  df_vendas.produto_id == df_estoque.produto_id,  "inner")
    .select(
        # Identificadores
        df_vendas["venda_id"],
        df_vendas["cliente_id"],
        df_vendas["produto_id"],

        # Dados do cliente
        df_clientes["nome_cliente"],
        df_clientes["cidade"],
        df_clientes["estado"],
        df_clientes["canal_aquisicao"],

        # Dados do produto
        df_estoque["nome_produto"],
        df_estoque["categoria"],
        df_estoque["custo_producao"],

        # Dados da venda
        df_vendas["quantidade"],
        df_vendas["preco_unitario"],
        df_vendas["desconto_percent"],
        df_vendas["forma_pagamento"],
        df_vendas["data_venda"],
        df_vendas["status_entrega"],
        df_vendas["ano_mes_data_venda"],
        df_vendas["valor_liquido"],
        df_vendas["valor_total"],

        # Dados de despesa
        df_vendas["tipo_despesa"],
        df_vendas["valor_despesa"],
        df_vendas["data_despesa"],
        df_vendas["ano_mes_data_despesa"],
    )
)

# faturamento_mensal
df_faturamento_mensal = df_base \
    .groupBy("ano_mes_data_venda") \
    .agg(
        sum("valor_total").alias("faturamento_total"),
        sum("valor_despesa").alias("custo_total"),
        count("venda_id").alias("qtde_vendas"),
    )\
        .withColumn("margem", round(col("faturamento_total") - col("custo_total"), 4)) \
        .withColumn("margem_percentual", 
                    when(col("faturamento_total") != 0,
                          round(col("margem") / col("faturamento_total"),4))) \
    .orderBy("ano_mes_data_venda")
df_faturamento_mensal.write\
    .option("overwriteSchema", "true")\
    .format("delta") \
    .mode("overwrite")\
    .saveAsTable("gold.default.faturamento_mensal_loja")

# Top produtos
df_top_produtos = df_base \
    .groupBy("nome_produto", "produto_id", "categoria")\
    .agg(
        sum("valor_total").alias("faturamento_total"),
        sum(col("custo_producao") *col("quantidade")).alias("custo_total"),
        count("venda_id").alias("qtde_vendas"),
    )\
        .withColumn("margem", round(col("faturamento_total") - col("custo_total"), 4)) \
        .withColumn("margem_percentual", 
                    when(col("faturamento_total") != 0,
                          round(col("margem") / col("faturamento_total"),4))) \
    .orderBy(desc("faturamento_total"))
df_top_produtos.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true")\
    .saveAsTable("gold.default.top_produtos_loja")
display(df_top_produtos)

# Top Clientes
df_top_clientes = df_base \
    .groupBy("cliente_id", "nome_cliente", "estado")\
        .agg(
            sum("valor_total").alias("faturamento_total"),
            sum(col("custo_producao") *col("quantidade")).alias("custo_total"),
            count("venda_id").alias("qtde_vendas"),
        )\
            .withColumn(
                "ticket_medio",
                when(col("qtde_vendas") != 0,
                    col("faturamento_total") / col("qtde_vendas")
                ).otherwise(0)
            )\
        .orderBy(desc("faturamento_total"))
df_top_clientes.write \
    .option("overwriteSchema", "true")\
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.default.top_clientes_loja")
display(df_top_clientes)

# top_canal_aquisicao
df_top_canal_aquisicao = df_base \
.groupBy("canal_aquisicao")\
    .agg(
        sum("valor_total").alias("faturamento_total"),
        sum(col("custo_producao") *col("quantidade")).alias("custo_total"),
        count("venda_id").alias("qtde_vendas"),
    )\
    .withColumn("margem", round(col("faturamento_total") - col("custo_total"), 4)) \
    .withColumn("margem_percentual", 
                    when(col("faturamento_total") != 0,
                          round(col("margem") / col("faturamento_total"),4))) \
    .orderBy(desc("faturamento_total"))
df_top_canal_aquisicao.write \
.option("overwriteSchema", "true") \
.format("delta") \
.mode("overwrite") \
.saveAsTable("gold.default.top_canal_aquisicao_loja")
display(df_top_canal_aquisicao)

# faturamento estado. 
df_faturamento_estado = df_base \
    .groupBy("estado") \
    .agg(
        sum("valor_total").alias("faturamento_total"),
        count("venda_id").alias("qtde_vendas")
    ) \
    .orderBy(desc("faturamento_total"))

df_faturamento_estado.write \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.default.faturamento_estado_loja")
display(df_faturamento_estado)

#Ticket Medio total
df_kpi = df_base.agg(
    sum("valor_total").alias("faturamento_total"),
    count("venda_id").alias("qtde_vendas")
).withColumn(
    "ticket_medio",
    col("faturamento_total") / col("qtde_vendas")
)
df_kpi.write \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.default.kpi_loja")
display(df_kpi)